# 演習6 解答編 ―― 容量とバックプレッシャ

> まず `ex06_backpressure.ipynb` を自分で解いてから読んでください。

## 発展課題1 の解答 ―― メモリを使い切るまで何分か

作る側が 30 fps、受け取る側が 15 fps とします。差は毎秒15フレームです。

### (1) メモリが尽きるまで

```
1秒に増える量 = 15フレーム × 384KB = 5.76 MB
4GB = 4096 MB
4096 ÷ 5.76 ≒ 711 秒 ≒ 約 12 分
```

### (2) 遅れが増えていく様子を図で見る

数字の前に、何が起きているかを図で押さえます。
**1行が1フレームの一生**です。`C` で撮られてキューに入り、`S` でキューから取り出されます。
あいだの `-` が、そのフレームが**キューで待った時間**です。

（`S` は演習6-1のプログラムが測っている `delay` と同じ点です。
`q.pop()` が返った瞬間で、そのあとの表示処理そのものは、この話の中では誤差です。）

```
【図1】作る 2枚/秒、出す 1枚/秒（比 2倍）
   1文字 = 0.5秒   C = 撮った   S = キューから出た   - = 順番待ち

F1    CS..............   待ち 0.5秒
F2    .C-S............   待ち 1.0秒
F3    ..C--S..........   待ち 1.5秒
F4    ...C---S........   待ち 2.0秒
F5    ....C----S......   待ち 2.5秒
F6    .....C-----S....   待ち 3.0秒
F7    ......C------S..   待ち 3.5秒
F8    .......C-------S   待ち 4.0秒
```

**`C` の列は1文字ずつしか下がらないのに、`S` の列は2文字ずつ逃げていきます。**
この差が開き続けることが、そのまま「遅れが増え続ける」ということです。

比を変えると、開き方が急になります。

```
【図2】作る 2枚/秒、出す 0.5枚/秒（比 4倍）
   1文字 = 0.5秒（図1と同じ目盛）

F1    C--S............   待ち 1.5秒
F2    .C-----S........   待ち 3.0秒
F3    ..C--------S....   待ち 4.5秒
F4    ...C-----------S   待ち 6.0秒
```

同じ8秒で、**図1は8枚さばけたのに図2は4枚**。そして F4 はすでに6秒待たされています。

### (3) 「2つの遅れ」がある

図1の右端（8秒の時点）を、縦に見てください。

```
  いま出ていくフレーム（F8）   … C は 4秒 ⇒ キューに 4秒いた
  いま入るフレーム（F16)       … これから 8秒 キューにいることになる
```

**どちらも同じ「キューに居た時間」で、見ているフレームが違うだけ**です。

- **画面の古さ** ＝ **いま出ていく1枚**の滞在時間
- **いま撮った1枚が出るまで** ＝ **いま入る1枚**の滞在時間

行列が伸び続けているので、**あとから入る枚ほど長く待たされます**。
だから同じ瞬間に測っても、2つの値が違います。

（キューの長さが一定なら、この2つは一致します。演習6-1の実測で `delay` が
安定した値になっていたのは、容量で頭打ちになっていたからです。）

### (4) 12分後の数字

では 30 fps → 15 fps に戻します。12分後、キューには
`15 × 711 ≒ 10,700 フレーム` 並んでいます。

```
いま撮った1枚が出るまで = 10,700 ÷ 15 ≒ 711 秒 ≒ 11.9 分
画面の古さ              = 711 ÷ 2      ≒ 356 秒 ≒  5.9 分
```

**「12分後には、いま撮った映像が画面に出るのは12分後」** ということです。

### (5) 遅れの増え方を式にする

一般化しておきます。動かして T 秒後、**そのとき入ったフレーム**の遅れは

```
溜まっている数 = (作る速さ − 受け取る速さ) × T
遅れ           = 溜まっている数 ÷ 受け取る速さ
               = ( 作る速さ ÷ 受け取る速さ − 1 ) × T
```

**遅れは経過時間 T に比例して増え、その係数は「速さの比」だけで決まります。**

- 30 fps → 15 fps（2倍） ⇒ 係数 **1.0**。1分動かせば1分遅れる（図1）
- 30 fps → 20 fps（1.5倍） ⇒ 係数 **0.5**。1分動かせば30秒遅れる
- 30 fps →  5 fps（6倍） ⇒ 係数 **5.0**。1分動かせば**5分**遅れる

「経過時間と同じだけ遅れる」のは、**ちょうど2倍のときだけ**です。

### (6) 受け取る側が遅いほど、早く落ちる

3つの場合を並べます。1フレーム384KB、メモリ上限4GBは共通です。

```
   作る→受け取る   溜まる速さ   落ちるまで   画面の古さ   いま撮った1枚が出るまで
   30 → 20 fps      3.8 MB/s     17.8 分       5.9 分            8.9 分
   30 → 15 fps      5.8 MB/s     11.9 分       5.9 分           11.9 分
   30 →  5 fps      9.6 MB/s      7.1 分       5.9 分           35.6 分
```

受け取る側が遅くなると、**落ちるのは早くなり、キューの吐き出しにかかる時間は長くなります。**

そして **「画面の古さ」が3つとも 5.9分で同じ**なのが、この表の面白いところです。

```
キューの中身 ＝「直近“古さ”秒ぶんに撮った映像」
枚数 = 撮る速さ × 古さ
  ⇒ 古さ = メモリに入る枚数 ÷ 撮る速さ = 10,700 ÷ 30 ≒ 356秒 ≒ 5.9分
```

**メモリが決めているのは「画面の古さの上限」で、それは撮る速さだけで決まります。**
受け取る側の速さが変えるのは、そこに到達するまでの時間と、吐き出すのにかかる時間だけです。

### (7) 落ちるずっと前から壊れている

いちばん大事なのはここです。経過1分の時点で、画面の古さはすでに

```
   30 → 20 fps   20 秒
   30 → 15 fps   30 秒
   30 →  5 fps   50 秒
```

**`30 → 5` なら、走らせて1分でもう50秒遅れ**です。メモリが尽きる7分より、はるか手前で
リアルタイム処理としては使いものになっていません。

**「メモリはまだ余裕があるから大丈夫」ではありません。**
そして最初の10秒だけ見て「ちゃんと動いている」と判断すると、まず気づけません。

> **遅れは、メモリより先に壊れる。**

## 発展課題2 の解答 ―― バックプレッシャは上流へ伝わる

Infer が遅いと、`Read → Infer` のキューが満杯になります。
すると Read 係は `push` の中の `can_push_.wait(...)` で**待たされます**。

```
Read ──▶ [満杯のキュー] ──▶ Infer(遅い) ──▶ [空のキュー] ──▶ Show
  ↑                            ↑
  待たされる                    ここが原因
```

Read 係は「読む」という自分の仕事はできるのに、**置き場所がないので進めません。**
結果として Read 係は Infer のペースに合わせて動くことになります。

**これが「上流に伝わる」ということ**です。詰まりは、下流から上流へ順に伝わります。
段が5つあって最後の段が遅ければ、最終的には**全部の段**が最後の段のペースになります。

### これは良いことか、悪いことか

**良いことです。** 理由が2つあります。

- **メモリが増えません。** 上流が勝手に先へ進まないので、どこにも溜まりません
- **Read 係が止まっているあいだ、CPU が空きます。** 演習1で見たとおり、
  コアは有限です。無駄に先へ進む仕事に取られるより、Infer に回るほうが良いことがあります

ただし、**上流が「止まれない」ものだと話が変わります。**
ファイルを読んでいるだけなら待てばよいのですが、
カメラやセンサは待ってくれません。読みに行かなければ、**そこで取りこぼします。**

その場合の対策が、次の発展課題3です。

## 発展課題3 の解答 ―― 古いものを捨てるという選択

**許される用途 ⇒ 「最新の1つだけに意味がある」もの**

- **画面表示**。5秒前のフレームをいま出しても意味がありません。最新を出すべきです
- **監視・モニタリング**。いまの温度・いまの速度が分かればよい
- **プレビュー**

**捨ててはいけない段 ⇒ 「すべてに意味がある」もの**

- **録画・保存**。1フレームでも欠けたら、それは別の動画です
- **集計・カウント**。数え落としは、そのまま結果の誤りになります
- **フレーム番号と結果を対応づけている処理**。
  捨てたことを下流が知らないと、**番号がずれて、別のフレームの結果が貼り付きます**

3つ目が特に危険です。「表示が少し飛ぶだけ」のつもりが、
検出枠が1つ前のフレームのものになる、という壊れ方をします。

次のセルで、待たせる場合と捨てる場合を比べます。
カメラは 30ms に1枚（止められない）、表示は1枚 60ms。表示は生成の半分の速さです。

In [ ]:
%%writefile ans06b.cpp
#include <iostream>
#include <iomanip>
#include <thread>
#include <queue>
#include <mutex>
#include <condition_variable>
#include <chrono>
using namespace std::chrono;

using TP = steady_clock::time_point;

// 満杯のとき「入れる側を待たせる」か「いちばん古いものを捨てる」かを選べるキュー
class DisplayQueue {
public:
    DisplayQueue(std::size_t cap, bool drop) : capacity_(cap), drop_(drop) {}

    void push(TP v) {
        std::unique_lock<std::mutex> lk(mtx_);
        if (drop_) {
            while (q_.size() >= capacity_) { q_.pop(); dropped_++; }   // 古いほうを捨てる
        } else {
            can_push_.wait(lk, [this] { return q_.size() < capacity_; });
        }
        q_.push(v);
        lk.unlock(); can_pop_.notify_one();
    }

    // 取り出せたら true。もう来ないなら false
    bool pop(TP& out) {
        std::unique_lock<std::mutex> lk(mtx_);
        can_pop_.wait(lk, [this] { return !q_.empty() || done_; });
        if (q_.empty()) return false;
        out = q_.front(); q_.pop();
        lk.unlock(); can_push_.notify_one();
        return true;
    }

    void set_done() {
        { std::lock_guard<std::mutex> g(mtx_); done_ = true; }
        can_pop_.notify_all();
    }
    long dropped() const { std::lock_guard<std::mutex> g(mtx_); return dropped_; }

private:
    std::queue<TP> q_;
    std::size_t capacity_;
    bool drop_;
    bool done_ = false;
    long dropped_ = 0;
    mutable std::mutex mtx_;
    std::condition_variable can_pop_, can_push_;
};

const int N = 60;
void wait_ms(int ms) { std::this_thread::sleep_for(milliseconds(ms)); }

void run(bool drop, const char* label) {
    DisplayQueue q(4, drop);
    long long stay_us = 0; int shown = 0;

    auto t0 = steady_clock::now();
    std::thread camera([&] { for (int i = 0; i < N; i++) { wait_ms(30); q.push(steady_clock::now()); } });
    std::thread display([&] {
        TP born;
        while (q.pop(born)) {
            stay_us += duration_cast<microseconds>(steady_clock::now() - born).count();
            shown++;
            wait_ms(60);
        }
    });
    camera.join();
    q.set_done();
    display.join();
    int ms = duration_cast<milliseconds>(steady_clock::now() - t0).count();

    std::cout << std::right << std::setw(8) << label
              << std::setw(11) << ms
              << std::setw(10) << shown
              << std::setw(9) << q.dropped()
              << std::setw(12) << (stay_us / (shown ? shown : 1) / 1000) << "\n";
}

int main() {
    std::cout << "カメラは 30ms に1枚（止められない）。表示は1枚 60ms かかる。\n";
    std::cout << "つまり表示は、生成の半分の速さしかない。キューの容量は 4。\n\n";
    std::cout << "  満杯時   time(ms)    表示数    捨てた   平均の遅れ(ms)\n";
    std::cout << "-----------------------------------------------------------\n";
    run(false, "block");     // 待たせる
    run(true,  "drop");      // 古いものを捨てる
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ans06b.cpp -o ans06b && ./ans06b

```
  満杯時   time(ms)    表示数    捨てた   平均の遅れ(ms)
-----------------------------------------------------------
   block       3642        60        0         248
    drop       2076        34       26         114
```

- **block（待たせる）** … 60枚すべて表示できますが、遅れが 248ms。
  そして、**この遅れは走らせるほど増え続けます**（発展課題1）
- **drop（捨てる）** … 26枚失いますが、遅れは半分以下で、**時間が経っても増えません**

どちらが正しいかは、**その段が何のためにあるか**で決まります。
表示なら drop、記録なら block です。

> **捨てるかどうかは性能の問題ではなく、仕様の問題。**

なお実装上の注意として、`drop` 版の `push` は
**満杯でも待ちません**（`can_push_.wait` を呼びません）。
待たないかわりに古いものを `pop()` して捨てています。
「入れる側は絶対に止めない」という約束のキューになっている、ということです。

## 発展課題4 の解答 ―― キューの長さでボトルネックを当てる

**言い当てられます。** これが本番でいちばん役に立つ道具かもしれません。

3段パイプラインで、キュー2本の長さを 5ms ごとに記録して平均を出します。
容量はどちらも4です。

In [ ]:
%%writefile ans06a.cpp
#include <iostream>
#include <iomanip>
#include <thread>
#include <atomic>
#include <queue>
#include <mutex>
#include <condition_variable>
#include <chrono>
using namespace std::chrono;

template <typename T>
class BoundedQueue {
public:
    explicit BoundedQueue(std::size_t capacity) : capacity_(capacity) {}
    void push(const T& v) {
        std::unique_lock<std::mutex> lk(mtx_);
        can_push_.wait(lk, [this] { return q_.size() < capacity_; });
        q_.push(v); lk.unlock(); can_pop_.notify_one();
    }
    T pop() {
        std::unique_lock<std::mutex> lk(mtx_);
        can_pop_.wait(lk, [this] { return !q_.empty(); });
        T v = q_.front(); q_.pop(); lk.unlock(); can_push_.notify_one();
        return v;
    }
    std::size_t size() const { std::lock_guard<std::mutex> g(mtx_); return q_.size(); }
private:
    std::queue<T> q_; std::size_t capacity_;
    mutable std::mutex mtx_;
    std::condition_variable can_pop_, can_push_;
};

const int N = 40;
const std::size_t CAP = 4;
void wait_ms(int ms) { std::this_thread::sleep_for(milliseconds(ms)); }

void run(int read_ms, int infer_ms, int show_ms, const char* which) {
    BoundedQueue<int> q1(CAP), q2(CAP);      // Read->Infer, Infer->Show
    std::atomic<bool> running{true};
    double sum1 = 0, sum2 = 0; long samples = 0;

    auto t0 = steady_clock::now();
    std::thread monitor([&] {
        while (running) { sum1 += q1.size(); sum2 += q2.size(); samples++; wait_ms(5); }
    });
    std::thread rd([&] { for (int i = 0; i < N; i++) { wait_ms(read_ms);  q1.push(i); } });
    std::thread inf([&] { for (int i = 0; i < N; i++) { int v = q1.pop(); wait_ms(infer_ms); q2.push(v); } });
    std::thread sh([&] { for (int i = 0; i < N; i++) { q2.pop(); wait_ms(show_ms); } });
    rd.join(); inf.join(); sh.join();
    running = false; monitor.join();
    int ms = duration_cast<milliseconds>(steady_clock::now() - t0).count();

    std::cout << std::right
              << std::setw(5) << read_ms << std::setw(7) << infer_ms << std::setw(6) << show_ms
              << std::setw(10) << ms
              << std::setw(11) << std::fixed << std::setprecision(1) << (sum1 / samples)
              << std::setw(11) << (sum2 / samples)
              << "   " << which << "\n";
}

int main() {
    std::cout << "3段パイプライン。キューは2本、どちらも容量 " << CAP << "。" << N << "フレーム流す。\n";
    std::cout << "q1 = Read から Infer へ、 q2 = Infer から Show へ\n\n";
    std::cout << " Read  Infer  Show  time(ms)  q1の長さ   q2の長さ   本当のボトルネック\n";
    std::cout << "----------------------------------------------------------------------\n";
    run(10, 60, 10, "Infer");
    run(10, 10, 60, "Show");
    run(60, 10, 10, "Read");
    run(30, 30, 30, "なし(横並び)");
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ans06a.cpp -o ans06a && ./ans06a

```
 Read  Infer  Show  time(ms)  q1の長さ   q2の長さ   本当のボトルネック
----------------------------------------------------------------------
   10     60    10      2431        3.7        0.0   Infer
   10     10    60      2427        3.1        3.7   Show
   60     10    10      2432        0.0        0.0   Read
   30     30    30      1270        0.0        0.0   なし(横並び)
```

きれいなパターンが出ています。

- **Infer が遅い** ⇒ q1 は満杯（3.7）、q2 は空（0.0）
- **Show が遅い** ⇒ q1 も q2 も満杯
- **Read が遅い** ⇒ どちらも空

読み取り方は1行にまとめられます。

> **満杯のキューのうち、いちばん下流のものの「すぐ次の段」が犯人。**
> **どのキューも空なら、犯人は一番上流の段。**

図で見ると、こういうことです。

```
Infer が遅い : Read ─[満杯]─ Infer ─[空]─ Show
                            ↑ 手前が詰まり、先は飢える

Show が遅い  : Read ─[満杯]─ Infer ─[満杯]─ Show
                                          ↑ 詰まりが上流まで届いている

Read が遅い  : Read ─[空]── Infer ─[空]── Show
                     ↑ そもそも供給が足りない
```

**詰まったキューの直後に、犯人がいます。** 詰まりは下流から上流へ伝わるので、
「満杯が続いている範囲の、いちばん下流の端」がボトルネックの位置です。

### (1) ただし、区別できないこともあります

上の表の3行目と4行目を見てください。
「Read が遅い」場合と「3段が横並び」の場合は、**どちらもキューが空**です。
キューの長さだけでは見分けられません。

見分けるには、**所要時間**を併せて見ます（2432ms と 1270ms）。
あるいは「一番上流の段が手待ちしていないか」を見ます。
上流が手待ちしていなければ、上流が全力で走ってなお足りない、ということです。

### (2) 実際にどう使うか

本番のプログラムで各段の時間をいちいち測るのは手間がかかります。
それより先に、**キューの長さを1行表示してみる**ほうが速いことがよくあります。

```cpp
std::cout << "q1=" << q1.size() << " q2=" << q2.size() << "\n";
```

数十行の計測コードを書く前に、この1行で当たりが付きます。

> **キューの長さは、それ自体がプロファイラである。**

（表示のために `size()` を呼ぶのは構いません。演習5の発展課題3で見たとおり、
やってはいけないのは**その値をもとに判断して動く**ことだけです。）

## 発展課題5 の解答 ―― 速くなったように見える測り方

容量を増やしても、スループットは変わりません（6-1）。
それでも「速くなった」という数字が出ることがあります。原因は主に3つです。

**① 途中で打ち切って測っている**

100個流すつもりで、60個目が出た時点で測定を止めたとします。
容量が大きければ、**まだキューの中に残っている仕事**があります。
それを数えずに「60個を◯秒で処理した」と言えば、当然速く見えます。

**やっていない仕事を、やったことにしている**わけです。

**② 最初の1個が出るまでの時間で測っている**

容量が大きいと、作る側は最初にどんどん先へ進めます。
「最初の1個が出てくるまで」だけを見れば、確かに速く見えます。
しかし**2個目以降のペースは変わりません。**

パイプラインには**立ち上がり**があります（演習1の発展課題1・2で見たとおりです）。
立ち上がりの区間だけを測ると、定常状態の速さは分かりません。

**③ 測っている区間が短すぎる**

キューが埋まりきる前に測定が終わってしまうと、
その間はバックプレッシャが一度も効きません。
容量が大きいほど「埋まりきるまで」が長いので、
短時間の測定では容量が大きいほうが有利に見えます。

### では、どう測るか

- **全部出終わるまで**を測る。途中で打ち切らない
- **立ち上がりを除いた区間**で測る。または十分に長く流して平均を取る
- 迷ったら、**キューの中身が最後に空になるまで**を1回分とする

> **「処理した個数 ÷ 時間」を測るときは、キューの中に残っているものを数え忘れない。**

これは本番でボトルネックを探すときに、そのまま効いてきます。
「改善した」と思ったものが、実は**仕事を後回しにしただけ**ということがよくあります。

---

## 参考：本番のプログラムでは

ハッカソンで読むプログラムでは、キュー2本の容量がソースの中で指定されています。

見るべき点は3つです。

1. **その容量は、何を根拠に決まっているのか**（たぶん、根拠はありません）
2. **各キューは、走らせているあいだ満杯なのか空なのか**
3. **満杯のキューの、すぐ下流の段は何か**

3番目が分かれば、ボトルネックの位置はほぼ確定します。
そこから先「その段をどう速くするか」は、本番の3日間で扱う話です。

なお、演習1で見たとおり、**ボトルネックは直すと移動します。**
1回見つけて終わりではなく、直すたびに測り直してください。